# Воспроизведение dspy.GEPA на AIME

Сетап туториала DSPy GEPA AIME (`dspy.ai/tutorials/gepa_aime`):
`dspy.ChainOfThought`, исполнитель gpt-4.1-mini (temperature=1, max_tokens=32000), рефлектор gpt-5,
метрика `int(answer) == N`, feedback с эталонным решением, `auto="light"` (около 560 вызовов метрики).
Данные: train 45 / val 45 / test 30 задач AIME 2025.

**Отличие от туториала:** кэш DSPy выключен (`cache=False`). В туториале `test = 30 задач × 5`,
и при включённом кэше все 5 повторов получают один и тот же ответ (оценка 150 примеров занимает около 1 секунды).
Поэтому 46,7% там на деле 14 из 30 задач, а не 70 независимых попыток. Здесь 5 повторов — это 5 настоящих сэмплов.

**Запуск**:
1. Установить зависимости: `uv venv --python 3.12 && source .venv/bin/activate && uv pip install --python .venv/bin/python gepa==0.1.4 dspy==3.4.1 python-dotenv requests datasets`.
2. Положить ключ OPENROUTER в файл `.env` рядом с ноутбуком: `OPENROUTER_API_KEY=sk-or-...`.

In [1]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()
assert os.environ.get("OPENROUTER_API_KEY"), "Нужно добавить OPENROUTER_API_KEY=... в файл .env рядом с ноутбуком"

def key_info():
    r = requests.get(
        "https://openrouter.ai/api/v1/key",
        headers={"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"},
        timeout=30,
    )
    r.raise_for_status()
    j = r.json()
    return j.get("data", j)

def usage():
    return key_info().get("usage") or 0.0

def check_budget(min_remaining):
    remaining = key_info().get("limit_remaining")
    if remaining is not None and remaining < min_remaining:
        raise RuntimeError(f"На ключе осталось ${remaining:.2f} < ${min_remaining}. Пополни лимит или уменьши план.")

print(key_info())

{'label': 'sk-or-v1-6fc...aea', 'is_management_key': False, 'is_provisioning_key': False, 'limit': 10, 'limit_reset': None, 'limit_remaining': 0.11092170000000046, 'include_byok_in_limit': False, 'budgets': [{'scope': 'key_limit', 'guardrail_id': None, 'budget_id': None, 'limit_usd': 10, 'spend_usd': 9.8890783, 'remaining_usd': 0.11092170000000046, 'reset_interval': None, 'resets_at': None, 'include_byok': False}], 'effective_budget': {'scope': 'key_limit', 'guardrail_id': None, 'budget_id': None, 'limit_usd': 10, 'spend_usd': 9.8890783, 'remaining_usd': 0.11092170000000046, 'reset_interval': None, 'resets_at': None, 'include_byok': False}, 'usage': 9.8890783, 'usage_daily': 0, 'usage_weekly': 0.8978624, 'usage_monthly': 9.8890783, 'byok_usage': 0, 'byok_usage_daily': 0, 'byok_usage_weekly': 0, 'byok_usage_monthly': 0, 'is_free_tier': False, 'expires_at': '2026-11-18T09:27:00.001Z', 'creator_user_id': 'user_37CT9Gw4co885VJ8ogcZRqLALOx', 'organization_id': None, 'workspace_id': 'b783f64

In [2]:
TASK_LM = "openrouter/openai/gpt-4.1-mini"
REFLECTION_LM = "openrouter/openai/gpt-5"

SEEDS = [0]
BUDGET = None  # None означает auto="light", как в туториале; число задаёт max_metric_calls
N_SAMPLES = 5  #5  независимых ответов на задачу при оценке на test как в туториале
N_BOOT = 10_000
THREADS = 2
MIN_REMAINING = 3 # $; ниже этого остатка новые прогоны не стартуют

OUT = "runs"
os.makedirs(f"{OUT}/eval", exist_ok=True)

In [4]:
import random
import dspy
from datasets import load_dataset

def init_dataset():
    train_split = [
        dspy.Example({"problem": x["problem"], "solution": x["solution"], "answer": x["answer"]}).with_inputs("problem")
        for x in load_dataset("AI-MO/aimo-validation-aime")["train"]
    ]
    random.Random(0).shuffle(train_split)
    test_split = [
        dspy.Example({"problem": x["problem"], "answer": x["answer"]}).with_inputs("problem")
        for x in load_dataset("MathArena/aime_2025")["train"]
    ]
    n = len(train_split)
    return train_split[: n // 2], train_split[n // 2 :], test_split

trainset, valset, testset = init_dataset()
N_TEST = len(testset)
print("train / val / test задач:", len(trainset), len(valset), N_TEST)

/Users/karimvafin/phd/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


train / val / test задач: 45 45 30


In [5]:
import json
import numpy as np
import pandas as pd

def to_matrix(records):
    M = np.full((N_TEST, N_SAMPLES), np.nan)
    for r in records:
        M[r["pid"], r["sample"]] = r["score"]
    assert not np.isnan(M).any(), "не все (задача, сэмпл) оценены"
    return M

def boot_ci(M, n_boot=N_BOOT, seed=0):
    per_problem = M.mean(axis=1)
    idx = np.random.default_rng(seed).integers(0, len(per_problem), (n_boot, len(per_problem)))
    boots = per_problem[idx].mean(axis=1)
    return per_problem.mean(), *np.percentile(boots, [2.5, 97.5])

def paired_diff_ci(A, B, n_boot=N_BOOT, seed=0):
    d = A.mean(axis=1) - B.mean(axis=1)
    idx = np.random.default_rng(seed).integers(0, len(d), (n_boot, len(d)))
    boots = d[idx].mean(axis=1)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return d.mean(), lo, hi, (boots <= 0).mean()

def fmt_ci(m, lo, hi):
    return f"{100*m:.1f}% [{100*lo:.1f}; {100*hi:.1f}]"

_rng = np.random.default_rng(1)
_M = (_rng.random((N_TEST, N_SAMPLES)) < _rng.random((N_TEST, 1))).astype(float)
print("пример ДИ:", fmt_ci(*boot_ci(_M)))

пример ДИ: 50.7% [38.0; 64.0]


In [6]:
task_lm = dspy.LM(TASK_LM, temperature=1.0, max_tokens=32000, cache=False, engine="litellm")
reflection_lm = dspy.LM(REFLECTION_LM, temperature=1.0, max_tokens=32000, cache=False, engine="litellm")
dspy.configure(lm=task_lm)

class GenerateResponse(dspy.Signature):
    """Solve the problem and provide the answer in the correct format."""
    problem = dspy.InputField()
    answer = dspy.OutputField()

def make_program():
    return dspy.ChainOfThought(GenerateResponse)

def metric(example, prediction, trace=None, pred_name=None, pred_trace=None):
    try:
        return int(int(example["answer"]) == int(prediction.answer))
    except (ValueError, TypeError, AttributeError):
        return 0

def metric_with_feedback(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = int(example["answer"])
    written_solution = example.get("solution", "")
    try:
        llm_answer = int(prediction.answer)
    except (ValueError, TypeError):
        feedback = (f"The final answer must be a valid integer and nothing else. You responded with "
                    f"'{prediction.answer}', which couldn't be parsed as a python integer. Please ensure your "
                    f"answer is a valid integer without any additional text or formatting. "
                    f"The correct answer is '{correct_answer}'.")
        if written_solution:
            feedback += (f" Here's the full step-by-step solution:\n{written_solution}\n\nThink about what takeaways "
                         "you can learn from this solution to improve your future answers and approach to similar "
                         "problems and ensure your final answer is a valid integer.")
        return dspy.Prediction(score=0, feedback=feedback)

    score = int(correct_answer == llm_answer)
    feedback = f"Your answer is {'correct' if score else 'incorrect'}. The correct answer is '{correct_answer}'."
    if written_solution:
        feedback += (f" Here's the full step-by-step solution:\n{written_solution}\n\nThink about what takeaways "
                     "you can learn from this solution to improve your future answers and approach to similar problems.")
    return dspy.Prediction(score=score, feedback=feedback)

def evaluate_on_test(program, tag):
    path = f"{OUT}/eval/{tag}.json"
    if os.path.exists(path):
        return json.load(open(path))
    partial_path = f"{OUT}/eval/{tag}.partial.json"
    done = {(r["pid"], r["sample"]): r for r in json.load(open(partial_path))} if os.path.exists(partial_path) else {}
    devset = [ex.copy(pid=pid, sample=k) for k in range(N_SAMPLES) for pid, ex in enumerate(testset)
              if (pid, k) not in done]
    if done:
        print(f"[{tag}] уже посчитано {len(done)}, осталось {len(devset)}")
    if devset:
        evaluate = dspy.Evaluate(devset=devset, metric=metric, num_threads=THREADS,
                                 display_progress=True, max_errors=len(devset))
        res = evaluate(program)
        for ex, pred, score in res.results:
            if "answer" in pred:
                done[(ex.pid, ex.sample)] = {"pid": ex.pid, "sample": ex.sample, "answer": str(pred.answer),
                                             "reasoning": str(getattr(pred, "reasoning", "")), "score": float(score)}
        os.makedirs(os.path.dirname(path), exist_ok=True)
        json.dump(list(done.values()), open(partial_path, "w"), ensure_ascii=False)
    n_total = N_SAMPLES * len(testset)
    if len(done) < n_total:
        raise RuntimeError(f"[{tag}] {n_total - len(done)}/{n_total} вызовов упали (лимит ключа? сеть?). "
                           f"Успешные сохранены в {partial_path}; исправь причину и запусти заново, "
                           "досчитаются только упавшие.")
    records = sorted(done.values(), key=lambda r: (r["sample"], r["pid"]))
    json.dump(records, open(path, "w"), ensure_ascii=False)
    os.remove(partial_path)
    return records


In [7]:
evals = {}

if not os.path.exists(f"{OUT}/eval/seed_prompt.json"):
    check_budget(MIN_REMAINING)
before = usage()
evals["seed_prompt"] = evaluate_on_test(make_program(), "seed_prompt")
print(f"baseline: ${usage() - before:.2f}")
print("seed_prompt", fmt_ci(*boot_ci(to_matrix(evals["seed_prompt"]))))

baseline: $0.00
seed_prompt 48.0% [32.7; 62.7]


In [8]:
import litellm
from gepa.core.state import GEPAState

class GuardStopper:
    # Мягкая остановка GEPA между итерациями: мало денег на ключе или пошли ошибки API.
    # Состояние сохраняется в gepa_logs/gepa_state.bin, повторный запуск продолжит с этого места.
    def __init__(self, min_remaining, max_errors=5):
        self.min_remaining, self.max_errors = min_remaining, max_errors
        self.errors, self.reason = 0, None
        litellm.failure_callback = [self.on_failure]

    def on_failure(self, kwargs, completion_response, start_time, end_time):
        self.errors += 1

    def __call__(self, gepa_state):
        try:
            remaining = key_info().get("limit_remaining")
        except Exception:
            remaining = None
        if remaining is not None and remaining < self.min_remaining:
            self.reason = f"на ключе осталось ${remaining:.2f}"
        elif self.errors >= self.max_errors:
            self.reason = f"{self.errors} ошибок вызова API"
        return self.reason is not None

def best_so_far(seed):
    # Лучший кандидат из чекпоинта (средняя точность на val), даже если прогон не завершился
    state = GEPAState.load(f"{OUT}/seed{seed}/gepa_logs")
    val = [sum(s.values()) / len(s) if s else 0.0 for s in state.prog_candidate_val_subscores]
    best = max(range(len(val)), key=val.__getitem__)
    print(f"seed {seed}: кандидатов {len(val)}, вызовов метрики {state.total_num_evals}, "
          f"лучший #{best} с val {100 * val[best]:.1f}% (исходный: {100 * val[0]:.1f}%)")
    return next(iter(state.program_candidates[best].values()))

def run_gepa(seed):
    d = f"{OUT}/seed{seed}"
    prog_path = f"{d}/program.json"
    program = make_program()
    if os.path.exists(prog_path):
        program.load(prog_path)
        return program
    check_budget(MIN_REMAINING)
    os.makedirs(d, exist_ok=True)
    before = usage()
    budget = {"auto": "light"} if BUDGET is None else {"max_metric_calls": BUDGET}
    guard = GuardStopper(MIN_REMAINING, max_errors=20)
    optimizer = dspy.GEPA(
        metric=metric_with_feedback,
        **budget,
        num_threads=THREADS,
        track_stats=True,
        reflection_minibatch_size=3,
        reflection_lm=reflection_lm,
        seed=seed,
        log_dir=f"{d}/gepa_logs",
        gepa_kwargs={"stop_callbacks": [guard]},
    )
    optimized = optimizer.compile(program, trainset=trainset, valset=valset)
    if guard.reason:
        open(f"{d}/partial_best_prompt.txt", "w").write(optimized.predict.signature.instructions)
        raise RuntimeError(f"seed {seed}: GEPA остановлен досрочно ({guard.reason}). Лучший промпт на данный момент: "
                           f"{d}/partial_best_prompt.txt. Пополни ключ и запусти ноутбук заново — прогон продолжится.")
    optimized.save(prog_path)
    dr = optimized.detailed_results
    json.dump({"seed": seed, "cost": usage() - before, "best_idx": dr.best_idx,
               "val_aggregate_scores": dr.val_aggregate_scores,
               "total_metric_calls": dr.total_metric_calls},
              open(f"{d}/meta.json", "w"), indent=1)
    open(f"{d}/best_prompt.txt", "w").write(optimized.predict.signature.instructions)
    return optimized

for seed in SEEDS:
    evals[f"best_seed{seed}"] = evaluate_on_test(run_gepa(seed), f"best_seed{seed}")
    print(f"seed {seed} готов, всего потрачено с ключа: ${usage():.2f}")

2026/10/28 08:48:46 WARNING dspy.primitives.base_module: There is a mismatch of python version between saved model and current environment. You saved with `python==3.12`, but now you have `python==3.11`. This might cause errors or performance downgrade on the loaded model, please consider loading the model in the same environment as the saving environment.


seed 0 готов, всего потрачено с ключа: $9.89


In [12]:
# Промпт, опубликованный в туториале DSPy (ячейка 18), на тех же 30 × N_SAMPLES без кэша
def make_tutorial_program():
    program = make_program()
    instructions = open("prompts/tutorial_prompt.txt").read().strip()
    program.predict.signature = program.predict.signature.with_instructions(instructions)
    return program

if not os.path.exists(f"{OUT}/eval/tutorial_prompt.json"):
    check_budget(MIN_REMAINING)
before = usage()
evals["tutorial_prompt"] = evaluate_on_test(make_tutorial_program(), "tutorial_prompt")
print(f"tutorial_prompt: ${usage() - before:.2f}")
print("tutorial_prompt", fmt_ci(*boot_ci(to_matrix(evals["tutorial_prompt"]))))

Average Metric: 68.00 / 150 (45.3%): 100%|██████████| 150/150 [32:24<00:00, 12.96s/it]

2026/10/29 13:09:01 INFO dspy.evaluate.evaluate: Average Metric: 68 / 150 (45.3%)



tutorial_prompt: $0.91
tutorial_prompt 45.3% [30.7; 60.0]


In [9]:
rows, diffs = [], []
base = to_matrix(evals["seed_prompt"])
for tag, recs in sorted(evals.items()):
    M = to_matrix(recs)
    m, lo, hi = boot_ci(M)
    rows.append({"candidate": tag, "acc": m, "ci95": fmt_ci(m, lo, hi),
                 "solved_any": int((M.max(axis=1) > 0).sum()), "solved_all": int((M.min(axis=1) == 1).sum())})
    if tag != "seed_prompt":
        dm, dlo, dhi, p = paired_diff_ci(M, base)
        diffs.append({"candidate": tag, "diff_pp": round(100 * dm, 1),
                      "ci95_pp": f"[{100*dlo:.1f}; {100*dhi:.1f}]", "P(diff<=0)": p})

results = pd.DataFrame(rows)
print("Точность на test (30 задач × N_SAMPLES), 95% ДИ кластерным бутстрепом по задачам;")
print("solved_any / solved_all: задач, решённых хотя бы раз / во всех сэмплах")
display(results.drop(columns="acc"))

if diffs:
    print("\nОптимизированная программа минус исходная (парный бутстреп по задачам), в п.п.")
    display(pd.DataFrame(diffs))

    acc = results[results.candidate.str.startswith("best_seed")].acc * 100
    print(f"\nРазброс между сидами GEPA: {acc.mean():.1f} ± {acc.std():.1f}% (сидов: {len(acc)})")

results.to_csv(f"{OUT}/summary.csv", index=False)

Точность на test (30 задач × N_SAMPLES), 95% ДИ кластерным бутстрепом по задачам;
solved_any / solved_all: задач, решённых хотя бы раз / во всех сэмплах


,candidate,ci95,solved_any,solved_all
0,best_seed0,46.7% [32.0; 62.0],19,8
1,seed_prompt,48.0% [32.7; 62.7],19,8



Оптимизированная программа минус исходная (парный бутстреп по задачам), в п.п.


,candidate,diff_pp,ci95_pp,P(diff<=0)
0,best_seed0,-1.3,[-7.3; 4.7],0.6834



Разброс между сидами GEPA: 46.7 ± nan% (сидов: 1)


In [10]:
# Утечка обучающих ответов в промпт: какие ответы train встречаются в лучших промптах
import re

train_answers = {str(int(x["answer"])) for x in trainset}
test_answers = {str(int(x["answer"])) for x in testset}

for seed in SEEDS:
    path = f"{OUT}/seed{seed}/best_prompt.txt"
    if not os.path.exists(path):
        continue
    nums = set(re.findall(r"\b\d{2,3}\b", open(path).read()))
    print(f"seed{seed}: ответы train в промпте: {sorted(nums & train_answers, key=int)}; "
          f"совпадают с ответами test: {sorted(nums & test_answers, key=int)}")

seed0: ответы train в промпте: ['51', '242', '315', '321']; совпадают с ответами test: []
